<a href="https://colab.research.google.com/github/MariiaOsokina/Nebius_module4_Performance_Engineering/blob/main/spec_dec%2Bquantization_homework3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework Assignment: Advanced LLM Acceleration: Speculative Decoding & Quantization

**Target hardware:** 1x NVIDIA H100 80GB

**Base model:** `Qwen/Qwen3-8B`

## Objective

Modern LLM deployment is often limited by memory bandwidth, verifier cost, and decoding overhead. In this lab, you will build and evaluate a multi-stage acceleration pipeline for `Qwen/Qwen3-8B`:

- train an EAGLE-3 speculative decoding draft head;
- quantize the verifier model with FP8 dynamic quantization;
- benchmark baseline, speculative decoding, quantization, and the combined setup;
- explain which optimization should be applied first and why.

The main question for the assignment:

**Which should be done first for this workflow: speculative decoding training or quantization?**

Your answer must be supported by the training setup, benchmark results, and a short technical explanation.

---

## References

- Speculators library: <https://github.com/vllm-project/speculators>
- Offline EAGLE-3 training tutorial: <https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>
- FP8 dynamic quantization reference: <https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

---

## Required Library Versions

Use separate environments. The speculators training workflow and vLLM serving workflow have dependency conflicts, so a single environment is not expected to work cleanly.

| Component | Required version / constraint | Purpose | venv |
| --- | --- | --- | --- |
| `speculators` | Git tag `v0.5.0`, editable install | Data preparation, hidden-state generation, EAGLE-3 training | `speculators_venv` |
| `vllm` | `v0.20.0` | Serving and benchmark runtime | `vllm_venv` |
| `fastapi` | `<0.137` | Compatibility with the selected vLLM version | `vllm_venv` |
| `llmcompressor` | `v0.12.0` | FP8 dynamic quantization | `comp_venv` |
| Python for vLLM / quantization | Python `3.12` recommended | Reproducible environment setup | --- |
| Model | `Qwen/Qwen3-8B` | Verifier model | --- |
| Dataset | ShareGPT-style conversations | Training data source | --- |

Expected local environments:

- `speculators_venv`
- `vllm_venv`
- `comp_venv`

Do not submit the virtual environments.

Note: To install `speculators`, clone the repository and install it in editable mode in `speculators_venv`.

---

## Task 1: Environment & Data Orchestration

Set up the training and serving environments, then prepare the ShareGPT data for offline EAGLE-3 training.

Required work:

- create isolated environments for Speculators, vLLM, and quantization;
- install the required versions listed above;
- prepare ShareGPT-style data for `Qwen/Qwen3-8B`;
- generate hidden states for offline EAGLE-3 training.

Background:

Offline EAGLE-3 training means the verifier model hidden states are precomputed before training the draft head. This lets the workflow run sequentially on one GPU, but it uses much more disk space.

Online EAGLE-3 training computes hidden states during training. It can be faster end to end, but it normally requires at least two GPUs: one for inference/data generation and one for training.

Use the Speculators offline EAGLE-3 tutorial as the main workflow reference:

<https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>

You need to implement an offline EAGLE-3 training pipeline.

Hints:

- Watch local disk usage. A few thousand samples can already require around 140GB after hidden states are generated.
- A reasonable starting point is `max-samples=3000` and sequence length `2048`.
- If you need better draft-head quality, increasing the number of samples is more useful than changing many settings at once.
- Use the `scripts/launch_vllm.py` script to run the vLLM server.

Question: Why do hidden states require much more disk space than the original text dataset?

Troubleshooting hints:

- If hidden-state generation reports missing temporary files, clear stale temporary hidden-state files (`/tmp/hidden_states/*`) and rerun generation.
- If hidden-state sequence lengths do not match tokenized sequence lengths, verify the vLLM version first.
- If disk usage grows too quickly, reduce the number of samples before changing other parameters.

---

**Task 1 question — why do hidden states need much more disk than the text?**


The text dataset stores just a few integer token IDs per token — very compact.

Hidden states store, for every token, multiple float vectors: EAGLE-3 caches several verifier layers (here layers 2, 18, 33 + the last), and each is a 4096-dimensional vector. So per token you go from a handful of integers to thousands of floats across multiple layers — orders of magnitude more bytes. Multiplied over thousands of samples × thousands of tokens each, this grows from a few MB of text to tens or hundreds of GB.

# Task 2: Train an EAGLE-3 Draft Head

Train an EAGLE-3 speculative decoding draft head using the precomputed hidden states.

Required work:

- Use `Qwen/Qwen3-8B` as the verifier model for training.
- Save checkpoints under `output/checkpoints/`.
- Use the best checkpoint for serving and benchmarking.
- Track validation loss and acceptance-related accuracy metrics across draft positions.

Hints:

- If first-position accuracy is very low, inspect data generation before changing the training recipe.
- Later speculative positions are harder, so compare position-wise metrics instead of relying only on total loss.

Reference training result from one run:

| Metric | Value |
| --- | ---: |
| `val/loss_0_epoch` | `2.509` |
| `val/full_acc_0_epoch` | `0.463` |
| `val/cond_acc_0_epoch` | `0.463` |
| `val/loss_1_epoch` | `3.778` |
| `val/full_acc_1_epoch` | `0.181` |
| `val/cond_acc_1_epoch` | `0.364` |
| `val/loss_2_epoch` | `4.550` |
| `val/full_acc_2_epoch` | `0.069` |
| `val/cond_acc_2_epoch` | `0.320` |
| `val/loss_epoch` | `10.837` |
| Epoch | `4` |

Note: Epoch=4 means this is the 5th epoch, the index starts with 0.

Questions to answer:

1. What do `full_acc` and `cond_acc` measure in speculative decoding training?
2. Why does accuracy usually decrease for later speculative positions?
3. What would you change if the first-position accuracy is very low?

---


## Task 2 results — EAGLE-3 draft head training

Verifier: Qwen/Qwen3-8B. 5 epochs, offline (precomputed hidden states), 3000 samples,
seq-len 2048. Best checkpoint: `output/checkpoints/checkpoint_best` (→ epoch 4, the 5th
epoch), `val/loss_epoch = 10.826`. My final-epoch metrics reproduce the reference almost
exactly, confirming the pipeline is correct:

| Metric | My run | Reference |
| --- | ---: | ---: |
| `val/loss_0_epoch` | 2.506 | 2.509 |
| `val/full_acc_0_epoch` | 0.463 | 0.463 |
| `val/cond_acc_0_epoch` | 0.463 | 0.463 |
| `val/loss_1_epoch` | 3.773 | 3.778 |
| `val/full_acc_1_epoch` | 0.181 | 0.181 |
| `val/cond_acc_1_epoch` | 0.365 | 0.364 |
| `val/loss_2_epoch` | 4.547 | 4.550 |
| `val/full_acc_2_epoch` | 0.068 | 0.069 |
| `val/cond_acc_2_epoch` | 0.321 | 0.320 |
| `val/loss_epoch` | 10.826 | 10.837 |
| Epoch | 4 (5th) | 4 |


## Task 2 questions

1. **What do full_acc and cond_acc measure?**
   The draft guesses several tokens ahead (positions 0, 1, 2…). The verifier checks them
   left to right, and the chain breaks at the first wrong guess. The two metrics count
   accuracy differently:
   - `cond_acc_i` = if the chain reached position i, how often the guess *at* i is correct —
     it **judges that one step alone**, assuming earlier ones were right.
   - `full_acc_i` = how often the **whole chain** survives: position i correct **and** every
     position before it correct. This is what drives accepted length (how many tokens are
     actually accepted per step).
   - At position 0 there is nothing before it, so they're equal (`full_acc_0 = cond_acc_0 =
     0.463`). After that `full_acc` is smaller because it requires all earlier guesses too.
     My run: full_acc 0.463 / 0.181 / 0.068; cond_acc 0.463 / 0.365 / 0.321.

2. **Why does accuracy decrease for later positions?**
   Two effects stack: (a) later positions predict further into the future and build on the
   draft's own earlier guesses, **so errors compound**; (b) `full_acc` requires being right at
   0 *and* 1 *and* 2, so it multiplies in another <1 factor each step. Together these make
   it fall off fast: 0.46 → 0.18 → 0.07.

3. **What would you change if first-position accuracy is very low?**
   Position 0 is the easiest (no compounding), so if it's bad the problem is in the
   **data/hidden-state generation, not the training recipe**. We need to check:
   - vLLM version (hidden-
   state seq len must match token seq len),
   - aux-layer IDs used for extraction match those
   passed to training,
   - correct chat template / loss mask, and
   - clear stale
   `/tmp/hidden_states`.
   
   Only then we touch training — and the best lever is **more samples**,
   which helps draft quality more than hyperparameter tweaks.

## Task 3: Quantize the Verifier Model

Quantize `Qwen/Qwen3-8B` using FP8 dynamic quantization.

Required work:

- Use `llmcompressor`.
- Apply FP8 dynamic quantization to linear layers.
- Keep `lm_head` unquantized.
- Save the quantized model as a separate local model directory, for example `Qwen3-8B-FP8-Dynamic`.
- Do not overwrite the original model checkpoint.

Reference material:

<https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

Hints:

- Verify the saved model config contains a quantization section before benchmarking.
- Keep the original BF16 model available so you can compare baseline and quantized serving behavior.

Expected quantization properties:

| Property | Expected value |
| --- | --- |
| Quantization method | compressed tensors |
| Weight format | FP8 |
| Activation format | dynamic FP8 |
| Target modules | linear layers |
| Ignored module | `lm_head` |

Questions to answer:

1. Why is FP8 dynamic quantization useful for serving on H100?
2. Why might `lm_head` be excluded from quantization?
3. How can quantization affect speculative decoding acceptance rate?

---

## Task 3 questions

1. **Why is FP8 dynamic quantization useful for serving on H100?**
   Decoding is memory-bandwidth bound; FP8 weights are half the bytes of BF16, so each
   decode step streams half the data from HBM → roughly double the headroom on the
   bottleneck. The H100 has native FP8 tensor cores, so there is no emulation penalty.
   "Dynamic" computes activation scales at runtime, so no calibration dataset is needed.

2. **Why might lm_head be excluded?**
   It maps to the full 151,936-token vocabulary and directly determines next-token
   probabilities, so it is precision-sensitive; **quantizing it risks quality loss for little speed benefit**. It is kept in BF16.

3. **How can quantization affect speculative decoding acceptance rate?**
   The draft head was trained against the BF16 verifier, so serving it on the FP8 verifier introduces a small train/serve mismatch: FP8 slightly changes the verifier's output distribution, so which drafted tokens get accepted can differ. In our runs the effect was negligible — comparing the *same* position, position-0 acceptance was essentially unchanged (BF16 K=2: 36.4%, FP8 K=1: 36.2%, FP8 K=2: 36.9%). The headline rates in the table (22.7% vs 36.2%) differ mainly because of the different K, not quantization: the K=2 figure averages in the much lower position-1 rate (~9%). So FP8 did not meaningfully hurt draft acceptance, but by making the verifier cheaper it does change the optimal
   draft-token count (K=2 → K=1).

## Task 4: Serve and Benchmark

Benchmark four configurations:

1. Baseline `Qwen/Qwen3-8B`
2. `Qwen/Qwen3-8B` with the trained EAGLE-3 draft head
3. FP8 dynamically quantized `Qwen3-8B`
4. FP8 dynamically quantized `Qwen3-8B` with the trained EAGLE-3 draft head

Required work:

- Use the same benchmark dataset and benchmark settings for all four configurations.
- Keep concurrency fixed across experiments.
- Disable prefix caching unless you intentionally study its effect.
- Use a fixed seed where possible.
- Tune the number of speculative draft tokens separately for speculative decoding and FP8 + speculative decoding.

Important:

The reference results used different numbers of speculative draft tokens for the unquantized speculative-decoding run and the FP8 + speculative-decoding run. Do not assume one value is optimal for both. Tune it yourself and justify the final choice using throughput, acceptance rate, acceptance length, and TPOT.

Hints:

- Start with a small number of speculative tokens, then increase only if the acceptance behavior supports it.
- Compare output token throughput first, then use TTFT, TPOT, and acceptance metrics to explain the result.
- If a setting produces more draft work but little accepted work, it is probably not the best setting.

Script for benchmarking:

```bash
vllm bench serve \
    --model Qwen/Qwen3-8B \
    --dataset-name hf \
    --max-concurrency 8 \
    --dataset-path philschmid/mt-bench \
    --num-prompts 80
```


Reference benchmark results:

| Configuration | Duration, s | Requests/s | Output tok/s | Total tok/s | Mean TTFT, ms | Mean TPOT, ms | Acceptance rate |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Baseline | `24.35` | `3.29` | `841.22` | `1090.87` | `576.17` | `7.28` | N/A |
| Speculative decoding | `16.27` | `4.92` | `1258.65` | `1632.19` | `78.17` | `5.76` | `22.48%` |
| FP8 quantization | `13.06` | `6.12` | `1566.56` | `2031.82` | `51.18` | `4.90` | N/A |
| FP8 + speculative decoding | `11.59` | `6.90` | `1766.55` | `2290.82` | `30.24` | `4.28` | `36.50%` |

Reference speculative decoding details:

| Configuration | Reference draft tokens | Acceptance length | Drafts | Draft tokens | Accepted tokens |
| --- | ---: | ---: | ---: | ---: | ---: |
| Speculative decoding | `2` | `1.45` | `14088` | `28176` | `6334` |
| FP8 + speculative decoding | `1` | `1.36` | `14954` | `14954` | `5458` |

Your exact numbers may differ, but the relative comparison should be explainable.

Questions to answer:

1. Why can speculative decoding improve throughput even when acceptance rate is not close to 100%?
2. How many speculative tokens are optimal for this setup? Explain using acceptance rate, acceptance length, and TPOT.

---

## Final Report Requirements

Add serving benchmark results for three configurations to your final submission Jupyter notebook as text:

- speculative decoding;
- FP8 quantization;
- FP8 + speculative decoding.

---

## Scoring Rubric

Scores are based on `Output token throughput (tok/s)` from `vllm bench serve`.
Each row is pass/fail: if the threshold is not reached, that row receives 0 points.

| Requirement | Passing threshold | Points |
| --- | ---: | ---: |
| Speculative decoding result with trained EAGLE-3 draft head | `> 1250 tok/s` | 25 |
| FP8 dynamic quantization result | `> 1550 tok/s` | 10 |
| Best combined FP8 + speculative decoding result, with draft-token tuning and explanation | `> 1750 tok/s` | 15 |
| **Total** |  | **50** |




## Task 4 questions

1. **Why can spec-dec improve throughput even when acceptance rate is far below 100%?**
   The verifier checks K drafted tokens in a single forward pass. Any accepted token is a
   token produced without a separate, weight-streaming decode step. Since decoding is
   bandwidth-bound, even an average of 1.36–1.45 accepted tokens per verifier pass
   (acceptance length) raises throughput, despite per-token acceptance being only ~22–37%.

2. **How many speculative tokens are optimal, and why?**
   - **Plain (BF16 verifier): K=2** → 1287 tok/s. Drafting a 2nd token pays off because the
     verifier step is relatively expensive, so amortizing it over more drafted tokens helps.
   - **FP8 verifier: K=1** → 1802 tok/s (K=2 → 1736, worse). With a cheap FP8 verifier, the
     cost of drafting + verifying a 2nd token that is only ~9% likely to be accepted
     outweighs the small gain in accept length (1.46 vs 1.36) and the extra ~2× draft tokens
     (27,952 vs 14,991), so TPOT rises and net throughput falls.
   Justification metrics: throughput, acceptance rate, acceptance length, and TPOT all point
   to K=1 for FP8 and K=2 for BF16 — the optimum depends on verifier cost.

This is an example of the output we expect to be submitted. This is the result we get from the baseline model out of the box:
```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  24.35     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              3.29      
Output token throughput (tok/s):         841.22    
Peak output token throughput (tok/s):    1144.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1090.87   
---------------Time to First Token----------------
Mean TTFT (ms):                          576.17    
Median TTFT (ms):                        46.05     
P99 TTFT (ms):                           5677.04   
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          7.28      
Median TPOT (ms):                        7.01      
P99 TPOT (ms):                           11.66     
---------------Inter-token Latency----------------
Mean ITL (ms):                           7.28      
Median ITL (ms):                         7.00      
P99 ITL (ms):                            7.68      
==================================================

```

Speculative decoding benchmark results (num_speculative_tokens = 2):
```
============ Serving Benchmark Result ============
Successful requests:                     80
Maximum request concurrency:             8
Benchmark duration (s):                  15.90
Total input tokens:                      6078
Total generated tokens:                  20473
Request throughput (req/s):              5.03
Output token throughput (tok/s):         1287.28
Total token throughput (tok/s):          1669.44
---------------Time to First Token----------------
Mean TTFT (ms):                          89.75
Median TTFT (ms):                        25.16
P99 TTFT (ms):                           669.22
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          5.68
Median TPOT (ms):                        5.75
P99 TPOT (ms):                           6.44
---------------Speculative Decoding---------------
Acceptance rate (%):                     22.73
Acceptance length:                       1.45
Drafts:                                  14028
Draft tokens:                            28056
Accepted tokens:                         6378
Per-position acceptance (%): pos0 36.38, pos1 9.08
==================================================
```

FP8 quantization benchmark results:

```
============ Serving Benchmark Result ============
Successful requests:                     80
Maximum request concurrency:             8
Benchmark duration (s):                  12.81
Total input tokens:                      6078
Total generated tokens:                  20480
Output token throughput (tok/s):         1598.81
Total token throughput (tok/s):          2075.10
---------------Time to First Token----------------
Mean TTFT (ms):                          25.88
Median TTFT (ms):                        24.56
P99 TTFT (ms):                           38.16
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          4.90
Median TPOT (ms):                        4.89
P99 TPOT (ms):                           4.99
==================================================
```

FP8 + speculative decoding benchmark results  (num_speculative_tokens = 1):

```
============ Serving Benchmark Result ============
Successful requests:                     80
Maximum request concurrency:             8
Benchmark duration (s):                  11.36
Total input tokens:                      6078
Total generated tokens:                  20480
Request throughput (req/s):              7.04
Output token throughput (tok/s):         1802.24
Total token throughput (tok/s):          2337.11
---------------Time to First Token----------------
Mean TTFT (ms):                          22.34
Median TTFT (ms):                        19.64
P99 TTFT (ms):                           39.54
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          4.27
Median TPOT (ms):                        4.28
P99 TPOT (ms):                           4.71
---------------Speculative Decoding---------------
Acceptance rate (%):                     36.20
Acceptance length:                       1.36
Drafts:                                  14991
Draft tokens:                            14991
Accepted tokens:                         5427
Per-position acceptance (%): pos0 36.20
==================================================
```

### Draft-token tuning (FP8 + speculative decoding)

| num_speculative_tokens | Output tok/s | Acceptance length | Draft tokens | Accepted tokens | Mean TPOT (ms) |
| ---: | ---: | ---: | ---: | ---: | ---: |
| **1 (chosen)** | **1802.24** | 1.36 | 14,991 | 5,427 | 4.27 |
| 2 | 1736.61 | 1.46 | 27,952 | 6,462 | 4.39 |

K=2 accepts slightly more per step (1.46 vs 1.36) but drafts ~2× the tokens for only
~19% more accepted, raising TPOT and lowering throughput → **K=1 is optimal for FP8+spec**
(vs K=2 for the BF16 verifier). Tuning justified by throughput, acceptance length, and TPOT.

## MAIN QUESTION: quantization or speculative-decoding training first?

**Quantization is first.** Reasons, supported by the runs above:

### 1. It is far cheaper to obtain.
FP8 dynamic quantization is a one-shot, data-free conversion (10–15 min, no calibration
set, no training). The EAGLE-3 draft head, by contrast, requires generating hidden states
(tens of GB) and training for several epochs.

### 2. It gives the larger standalone win.
- FP8 alone: 841 → 1598 tok/s **(+90%)**.
- Spec-dec alone: 841 → 1287 tok/s **(+53%)**.

On an H100, decoding is memory-bandwidth bound, and halving the weight bytes attacks that
bottleneck directly — so quantization moves the dominant cost more than speculative
decoding does.

### 3. It changes how you must tune speculative decoding — so it belongs underneath it.
The optimal draft-token count K is **different on the quantized verifier (K=1) than on the
BF16 verifier (K=2)**. My sweep confirmed this: FP8+spec was best at K=1 (1802 tok/s) and
*worse* at K=2 (1736), whereas plain spec-dec was best at K=2 (1287).

Why the optimum shifts: choosing K is a cost/benefit trade-off. A 2nd draft token *might*
be accepted (benefit), but the draft must produce it and the verifier must check it every
step regardless (cost). Quantization changes the **cost** side — the FP8 verifier pass is
much cheaper (TPOT ~4.3 ms vs ~5.7 ms BF16):
- **Expensive BF16 verifier:** one verifier pass dominates, so it pays to amortize it over
  more drafted tokens → drafting a 2nd token is worth it even at ~9% acceptance → **K=2**.
- **Cheap FP8 verifier:** the verifier pass no longer dominates, so the extra draft+verify
  work for that rarely-accepted 2nd token is pure overhead → **K=1**. My data shows it: at
  K=2, 27,952 drafted to accept 6,462; at K=1, only 14,991 drafted to accept 5,427 — ~2×
  the draft work for ~19% more accepted tokens, which *raised* TPOT and *lowered*
  throughput.

This makes the ordering a dependency, not just a preference: spec-dec tuning is *downstream*
of the verifier's cost profile. If you trained/tuned spec-dec first (concluding K=2) and
then quantized, the K=2 conclusion would be stale and you'd have to re-sweep K anyway.
Quantize first → that becomes the final verifier → train and tune the draft head (and K)
against it once.

### Conclusion
The two optimizations are complementary and multiplicative (1802 > either alone), but the
correct ordering is **quantize → then add and tune the EAGLE-3 draft head**.
